# Figure 5 | Disease-associated glial states in vulnerable NVUs

Publication-ready analysis notebook for the manuscript figure. Exploratory checks, scratch outputs, and analyses unrelated to the figure panels have been removed. Paths are repository-relative; set `NVU_PROJECT_ROOT` when running from another location.

Analysis scope:
- Astrocyte and microglial subtype composition in AD-associated NVUs.
- DAA/DAM density, proportion, and selected-region summaries used for manuscript panels.

In [ ]:
# Repository-relative paths
PROJECT_ROOT <- Sys.getenv("NVU_PROJECT_ROOT", unset = normalizePath("..", mustWork = FALSE))
DATA_DIR <- file.path(PROJECT_ROOT, "data")
RESULTS_DIR <- file.path(PROJECT_ROOT, "results")
REFERENCE_DIR <- file.path(PROJECT_ROOT, "references")
FIGURE_DIR <- file.path(RESULTS_DIR, "figure5")
dir.create(FIGURE_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
suppressMessages({library(Seurat)
library(tidyverse)
library(cowplot)
library(patchwork)
library(ggplot2)
#library(ggthemes)
library(plyr)
library(WGCNA)
library(hdWGCNA)
library(harmony)
library(corrplot)
library(igraph)
library(UCell)
theme_set(theme_cowplot())})
set.seed(123)
suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
})

out_dir <- "../results/08_DAA_DAM/"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
  library(scales)
})

# 0. Paths

out_dir <- "../results/08_DAA_DAM/subtype_donut_scaled"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# hip_rds <- "../results/05_nvu_analysis_results/Hip/200/Hip_merged-subtype_0330.rds"
# cortex_rds <- "../results/05_nvu_analysis_results/Cortex/200/Cortex_merged-subtype_0330.rds"

# 1. Read Seurat objects

NVU_hip <- readRDS(hip_rds)
NVU_cortex <- readRDS(cortex_rds)

# 2. Helper functions

repair_colnames <- function(df) {
  nm <- names(df)
  bad <- is.na(nm) | nm == ""
  nm[bad] <- paste0("blank_col_", seq_len(sum(bad)))
  names(df) <- make.unique(nm, sep = "_")
  df
}

std_group <- function(x) {
  x <- as.character(x)
  dplyr::case_when(
    stringr::str_detect(tolower(x), "^ad") ~ "AD",
    stringr::str_detect(tolower(x), "con|ctrl|control") ~ "Control",
    TRUE ~ x
  )
}

as_bool <- function(x) {
  if (is.logical(x)) return(x)
  tolower(as.character(x)) %in% c("true", "t", "1", "yes", "y")
}

merge_cortex_region <- function(x) {
  region_map <- c(
    L1 = "L1",
    L2 = "L23", L23 = "L23", L3 = "L23",
    L4 = "L456", L45 = "L456", L456 = "L456", L5 = "L456", L6 = "L456",
    WM = "WM"
  )
  y <- as.character(x)
  idx <- y %in% names(region_map)
  y[idx] <- unname(region_map[y[idx]])
  y
}

to_meta_df <- function(obj, tissue_name) {
  df <- if (inherits(obj, "Seurat")) obj@meta.data else as.data.frame(obj)
  df <- repair_colnames(df)

  need_cols <- c("group", "sample_id", "area_m", "subcelltype", "unit_id", "dist")
  miss_cols <- setdiff(need_cols, colnames(df))
  if (length(miss_cols) > 0) {
    stop("Missing columns in metadata: ", paste(miss_cols, collapse = ", "))
  }

  df <- df %>%
    dplyr::mutate(
      tissue = tissue_name,
      group = std_group(group),
      sample_id = as.character(sample_id),
      area_m = as.character(area_m),
      subcelltype = as.character(subcelltype),
      unit_id = as.character(unit_id),
      dist = as_bool(dist)
    ) %>%
    dplyr::filter(
      group %in% c("AD", "Control"),
      dist == TRUE,
      !is.na(unit_id),
      unit_id != "",
      !is.na(area_m),
      !is.na(subcelltype)
    )

  if (tissue_name == "Cortex") {
    df$area_m <- merge_cortex_region(df$area_m)
  }

  df$unit_uid <- paste(df$tissue, df$sample_id, df$unit_id, sep = "__")
  df
}

calc_mean_count_per_unit <- function(meta_df, prefix) {
  subtypes <- sort(unique(meta_df$subcelltype[stringr::str_detect(meta_df$subcelltype, paste0("^", prefix, "_"))]))

  units <- meta_df %>%
    dplyr::distinct(tissue, group, sample_id, area_m, unit_uid)

  unit_counts <- meta_df %>%
    dplyr::filter(subcelltype %in% subtypes) %>%
    dplyr::count(tissue, group, sample_id, area_m, unit_uid, subcelltype, name = "count") %>%
    dplyr::mutate(subtype = subcelltype) %>%
    dplyr::select(-subcelltype)

  unit_grid <- tidyr::crossing(units, subtype = subtypes)

  unit_counts_full <- unit_grid %>%
    dplyr::left_join(
      unit_counts,
      by = c("tissue", "group", "sample_id", "area_m", "unit_uid", "subtype")
    ) %>%
    dplyr::mutate(count = tidyr::replace_na(count, 0))

  sample_mean <- unit_counts_full %>%
    dplyr::group_by(tissue, sample_id, area_m, group, subtype) %>%
    dplyr::summarise(
      mean_count_per_sample = mean(count, na.rm = TRUE),
      n_units_per_sample = dplyr::n_distinct(unit_uid),
      .groups = "drop"
    )

  group_mean <- sample_mean %>%
    dplyr::group_by(tissue, area_m, group, subtype) %>%
    dplyr::summarise(
      mean_count = mean(mean_count_per_sample, na.rm = TRUE),
      sd_count = sd(mean_count_per_sample, na.rm = TRUE),
      n_samples = dplyr::n_distinct(sample_id),
      .groups = "drop"
    )

  list(sample_mean = sample_mean, group_mean = group_mean)
}

make_colors <- function(base_colors, subtype_order, control_alpha = 0.35) {
  color_df <- expand.grid(
    subtype = subtype_order,
    group = c("Control", "AD"),
    stringsAsFactors = FALSE
  )

  color_df$celltype_group <- paste(color_df$subtype, color_df$group, sep = "_")
  color_df$color <- ifelse(
    color_df$group == "AD",
    base_colors[color_df$subtype],
    scales::alpha(base_colors[color_df$subtype], control_alpha)
  )

  stats::setNames(color_df$color, color_df$celltype_group)
}

make_labels <- function(label_map, subtype_order) {
  out <- c()
  for (st in subtype_order) {
    out[paste0(st, "_Control")] <- paste0(label_map[[st]], " (Control)")
    out[paste0(st, "_AD")] <- paste0(label_map[[st]], " (AD)")
  }
  out
}

prepare_scaled_donut_data <- function(mean_df, tissue_use, subtype_order) {
  plot_df <- mean_df %>%
    dplyr::filter(tissue == tissue_use, subtype %in% subtype_order) %>%
    dplyr::transmute(
      area_m = as.character(area_m),
      group = factor(as.character(group), levels = c("Control", "AD")),
      subtype = factor(as.character(subtype), levels = subtype_order),
      mean_count = mean_count
    )

  full_grid <- plot_df %>%
    dplyr::distinct(area_m) %>%
    tidyr::crossing(
      subtype = factor(subtype_order, levels = subtype_order),
      group = factor(c("Control", "AD"), levels = c("Control", "AD"))
    )

  plot_df <- full_grid %>%
    dplyr::left_join(plot_df, by = c("area_m", "subtype", "group")) %>%
    dplyr::mutate(
      mean_count = tidyr::replace_na(mean_count, 0),
      celltype_group = paste(as.character(subtype), as.character(group), sep = "_"),
      subtype_i = as.integer(subtype),
      group_i = as.integer(group)
    ) %>%
    dplyr::arrange(area_m, subtype_i, group_i)

  area_total <- plot_df %>%
    dplyr::group_by(area_m) %>%
    dplyr::summarise(
      area_total_mean = sum(mean_count, na.rm = TRUE),
      .groups = "drop"
    )

  max_area_total <- max(area_total$area_total_mean, na.rm = TRUE)
  if (!is.finite(max_area_total) || max_area_total <= 0) max_area_total <- 1

  plot_df %>%
    dplyr::left_join(area_total, by = "area_m") %>%
    dplyr::group_by(area_m) %>%
    dplyr::mutate(
      scaled_frac = mean_count / max_area_total,
      ymax = cumsum(scaled_frac),
      ymin = dplyr::lag(ymax, default = 0),
      xmin = 1.35,
      xmax = 2.35,
      scaled_total = area_total_mean / max_area_total
    ) %>%
    dplyr::ungroup() %>%
    dplyr::select(-subtype_i, -group_i)
}

plot_scaled_donut <- function(
  donut_df,
  region_order,
  colors_vector,
  label_vector,
  title_text,
  file_prefix,
  ncol_plot = 4,
  width = 7,
  height = 4.5
) {
  celltype_group_order <- names(colors_vector)

  plot_df <- donut_df %>%
    dplyr::mutate(
      area_m = factor(area_m, levels = region_order),
      celltype_group = factor(celltype_group, levels = celltype_group_order)
    ) %>%
    dplyr::filter(!is.na(area_m))

  bg_df <- plot_df %>%
    dplyr::distinct(area_m, xmin, xmax) %>%
    dplyr::mutate(ymin = 0, ymax = 1)

  p <- ggplot() +
    geom_rect(
      data = bg_df,
      aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax),
      fill = "white",
      colour = NA
    ) +
    geom_rect(
      data = plot_df,
      aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = celltype_group),
      colour = "white",
      linewidth = 0.35
    ) +
    scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
    coord_polar(theta = "y") +
    xlim(c(0.50, 2.50)) +
    facet_wrap(~ area_m, ncol = ncol_plot) +
    scale_fill_manual(
      values = colors_vector,
      breaks = celltype_group_order,
      labels = label_vector[celltype_group_order],
      drop = FALSE,
      name = "Subtype"
    ) +
    labs(
      title = title_text,
      subtitle = "Largest regional total = full ring; each subtype is paired as Control then AD",
      caption = "Arc length indicates mean cell count per digital NVU scaled by the maximum regional total."
    ) +
    theme_void(base_size = 8, base_family = "Arial") +
    theme(
      plot.title = element_text(face = "bold", hjust = 0.5, size = 11),
      plot.subtitle = element_text(hjust = 0.5, size = 6.8, colour = "white"),
      plot.caption = element_text(hjust = 0.5, size = 5.8, colour = "white"),
      strip.text = element_text(size = 8.5, face = "bold", colour = "black"),
      legend.position = "bottom",
      legend.title = element_text(size = 6.5, face = "bold"),
      legend.text = element_text(size = 5.5),
      legend.key.size = unit(3, "mm"),
      legend.spacing.x = unit(0.7, "mm"),
      panel.spacing = unit(1.3, "lines"),
      plot.margin = margin(4, 4, 4, 4)
    ) +
    guides(fill = guide_legend(ncol = 5, byrow = TRUE, override.aes = list(colour = NA)))

  ggsave(file.path(out_dir, paste0(file_prefix, ".pdf")), p, width = width, height = height, device = cairo_pdf)
  ggsave(file.path(out_dir, paste0(file_prefix, ".png")), p, width = width, height = height, dpi = 600, bg = "white")

  p
}

# 3. Metadata and mean count

NVU_hip_meta <- to_meta_df(NVU_hip, "Hippocampus")
NVU_cortex_meta <- to_meta_df(NVU_cortex, "Cortex")
meta_all <- dplyr::bind_rows(NVU_hip_meta, NVU_cortex_meta)

astro_res <- calc_mean_count_per_unit(meta_all, "Astro")
micro_res <- calc_mean_count_per_unit(meta_all, "Micro")

astro_mean <- astro_res$group_mean
micro_mean <- micro_res$group_mean

write.csv(astro_mean, file.path(out_dir, "Astro_group_mean_count_per_unit.csv"), row.names = FALSE)
write.csv(micro_mean, file.path(out_dir, "Micro_group_mean_count_per_unit.csv"), row.names = FALSE)

# 4. Plot settings

astro_order <- paste0("Astro_", 0:4)
micro_order <- paste0("Micro_", 0:4)

astro_labels <- c(
  Astro_0 = "Astro.0",
  Astro_1 = "Astro.1",
  Astro_2 = "DAA.1",
  Astro_3 = "DAA.2",
  Astro_4 = "Astro.4"
)

micro_labels <- c(
  Micro_0 = "Micro.0",
  Micro_1 = "Micro.1",
  Micro_2 = "DAM",
  Micro_3 = "Micro.3",
  Micro_4 = "Micro.4"
)

astro_base_colors <- c(
  Astro_0 = "#2F5597",
  Astro_1 = "#56B4E9",
  Astro_2 = "#E69F00",
  Astro_3 = "#CC79A7",
  Astro_4 = "#009E73"
)

micro_base_colors <- c(
  Micro_0 = "#6A3D9A",
  Micro_1 = "#1B9E77",
  Micro_2 = "#D55E00",
  Micro_3 = "#7570B3",
  Micro_4 = "#999999"
)

astro_colors <- make_colors(astro_base_colors, astro_order, control_alpha = 0.35)
micro_colors <- make_colors(micro_base_colors, micro_order, control_alpha = 0.35)

astro_label_vector <- make_labels(astro_labels, astro_order)
micro_label_vector <- make_labels(micro_labels, micro_order)

# 5. Build donut data

hip_regions <- c("CA1", "CA2", "CA3", "CA4", "DG", "FAS", "SLRM")
cortex_regions <- c("L1", "L23", "L456", "WM")

astro_hip_donut <- prepare_scaled_donut_data(astro_mean, "Hippocampus", astro_order)
astro_cortex_donut <- prepare_scaled_donut_data(astro_mean, "Cortex", astro_order)

micro_hip_donut <- prepare_scaled_donut_data(micro_mean, "Hippocampus", micro_order)
micro_cortex_donut <- prepare_scaled_donut_data(micro_mean, "Cortex", micro_order)

write.csv(astro_hip_donut, file.path(out_dir, "Astro_Hippocampus_scaled_donut_data.csv"), row.names = FALSE)
write.csv(astro_cortex_donut, file.path(out_dir, "Astro_Cortex_scaled_donut_data.csv"), row.names = FALSE)
write.csv(micro_hip_donut, file.path(out_dir, "Micro_Hippocampus_scaled_donut_data.csv"), row.names = FALSE)
write.csv(micro_cortex_donut, file.path(out_dir, "Micro_Cortex_scaled_donut_data.csv"), row.names = FALSE)

# 6. Draw plots

p_astro_hip <- plot_scaled_donut(
  astro_hip_donut,
  hip_regions,
  astro_colors,
  astro_label_vector,
  "Astrocyte subtype composition per digital NVU",
  "Astro_Hippocampus_scaled_donut",
  ncol_plot = 4,
  width = 7,
  height = 4.5
)

p_astro_cortex <- plot_scaled_donut(
  astro_cortex_donut,
  cortex_regions,
  astro_colors,
  astro_label_vector,
  "Astrocyte subtype composition per digital NVU",
  "Astro_Cortex_scaled_donut",
  ncol_plot = 4,
  width = 6,
  height = 3.6
)

p_micro_hip <- plot_scaled_donut(
  micro_hip_donut,
  hip_regions,
  micro_colors,
  micro_label_vector,
  "Microglia subtype composition per digital NVU",
  "Micro_Hippocampus_scaled_donut",
  ncol_plot = 4,
  width = 7,
  height = 4.5
)

p_micro_cortex <- plot_scaled_donut(
  micro_cortex_donut,
  cortex_regions,
  micro_colors,
  micro_label_vector,
  "Microglia subtype composition per digital NVU",
  "Micro_Cortex_scaled_donut",
  ncol_plot = 4,
  width = 6,
  height = 3.6
)

print(p_astro_hip)
print(p_astro_cortex)
print(p_micro_hip)
print(p_micro_cortex)

cat("Done. Output directory:\n", out_dir, "\n")

In [ ]:
NVU_hip <- readRDS(
  "../results/05_nvu_analysis_results/Hip/200/Hip_merged-subtype_0330.rds"
)

NVU_cortex <- readRDS(
  "../results/05_nvu_analysis_results/Cortex/200/Cortex_merged-subtype_0330.rds"
)


NVU_hip <- readRDS(
  "../results/05_nvu_analysis_results/Hip/200/Hip_merged-subtype_0330.rds"
)

NVU_cortex <- readRDS(
  "../results/05_nvu_analysis_results/Cortex/200/Cortex_merged-subtype_0330.rds"
)   
                   
NVU_hip_meta <- to_meta_df(NVU_hip)
NVU_cortex_meta <- to_meta_df(NVU_cortex)
                   
NVU_cortex_meta <- subset(NVU_cortex_meta, subset = dist == TRUE)
NVU_hip_meta <- subset(NVU_hip_meta, subset = dist == TRUE)
# Significance statistics by region and subtype

calc_region_subtype_stats <- function(sample_mean_df, population_name, out_dir) {
  stat_df <- sample_mean_df %>%
    dplyr::group_by(tissue, area_m, subtype) %>%
    dplyr::summarise(
      n_AD = sum(group == "AD"),
      n_Control = sum(group == "Control"),

      mean_AD = mean(mean_count_per_sample[group == "AD"], na.rm = TRUE),
      mean_Control = mean(mean_count_per_sample[group == "Control"], na.rm = TRUE),

      median_AD = median(mean_count_per_sample[group == "AD"], na.rm = TRUE),
      median_Control = median(mean_count_per_sample[group == "Control"], na.rm = TRUE),

      sd_AD = stats::sd(mean_count_per_sample[group == "AD"], na.rm = TRUE),
      sd_Control = stats::sd(mean_count_per_sample[group == "Control"], na.rm = TRUE),

      sem_AD = sd_AD / sqrt(n_AD),
      sem_Control = sd_Control / sqrt(n_Control),

      delta_AD_minus_Control = mean_AD - mean_Control,
      log2FC_AD_vs_Control = log2((mean_AD + 1e-6) / (mean_Control + 1e-6)),

      p_wilcox = tryCatch({
        x <- mean_count_per_sample[group == "AD"]
        y <- mean_count_per_sample[group == "Control"]
        if (length(x) >= 2 && length(y) >= 2) {
          stats::wilcox.test(x, y, exact = FALSE)$p.value
        } else {
          NA_real_
        }
      }, error = function(e) NA_real_),

      p_ttest = tryCatch({
        x <- mean_count_per_sample[group == "AD"]
        y <- mean_count_per_sample[group == "Control"]
        if (length(x) >= 2 && length(y) >= 2) {
          stats::t.test(x, y)$p.value
        } else {
          NA_real_
        }
      }, error = function(e) NA_real_),

      .groups = "drop"
    ) %>%
    dplyr::group_by(tissue) %>%
    dplyr::mutate(
      p_adj_wilcox_BH = stats::p.adjust(p_wilcox, method = "BH"),
      p_adj_ttest_BH = stats::p.adjust(p_ttest, method = "BH")
    ) %>%
    dplyr::ungroup() %>%
    dplyr::mutate(
      population = population_name,
      sig = dplyr::case_when(
        !is.na(p_adj_wilcox_BH) & p_adj_wilcox_BH < 0.001 ~ "***",
        !is.na(p_adj_wilcox_BH) & p_adj_wilcox_BH < 0.01  ~ "**",
        !is.na(p_adj_wilcox_BH) & p_adj_wilcox_BH < 0.05  ~ "*",
        TRUE ~ ""
      )
    ) %>%
    dplyr::select(
      population, tissue, area_m, subtype,
      n_AD, n_Control,
      mean_AD, mean_Control,
      delta_AD_minus_Control,
      log2FC_AD_vs_Control,
      median_AD, median_Control,
      sem_AD, sem_Control,
      p_wilcox, p_adj_wilcox_BH,
      p_ttest, p_adj_ttest_BH,
      sig
    )

  write.csv(
    stat_df,
    file.path(out_dir, paste0(population_name, "_region_subtype_AD_vs_Control_stats.csv")),
    row.names = FALSE
  )

  stat_df
}

astro_stats <- calc_region_subtype_stats(
  astro_res$sample_mean,
  "Astro",
  out_dir
)

micro_stats <- calc_region_subtype_stats(
  micro_res$sample_mean,
  "Micro",
  out_dir
)

all_subtype_stats <- dplyr::bind_rows(astro_stats, micro_stats)

write.csv(
  all_subtype_stats,
  file.path(out_dir, "Astro_Micro_region_subtype_AD_vs_Control_stats.csv"),
  row.names = FALSE
)

sig_results <- all_subtype_stats %>%
  dplyr::filter(!is.na(p_adj_wilcox_BH), p_adj_wilcox_BH < 0.05) %>%
  dplyr::arrange(tissue, area_m, p_adj_wilcox_BH)

write.csv(
  sig_results,
  file.path(out_dir, "Significant_Astro_Micro_region_subtype_AD_vs_Control.csv"),
  row.names = FALSE
)

sig_results

In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
  library(scales)
})

# 0. Output path

out_dir <- "../results/08_DAA_DAM/subtype_barplot_AD_vs_Control"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

show_points <- FALSE

p_for_plot <- "p_adj_ttest_BH"

# 1. Read data

NVU_hip <- readRDS(
  "../results/05_nvu_analysis_results/Hip/200/Hip_merged-subtype_0330.rds"
)

NVU_cortex <- readRDS(
  "../results/05_nvu_analysis_results/Cortex/200/Cortex_merged-subtype_0330.rds"
)

# 2. Helper functions

repair_colnames <- function(df) {
  nm <- names(df)
  bad <- is.na(nm) | nm == ""
  nm[bad] <- paste0("blank_col_", seq_len(sum(bad)))
  names(df) <- make.unique(nm, sep = "_")
  df
}

std_group <- function(x) {
  x <- as.character(x)
  dplyr::case_when(
    stringr::str_detect(tolower(x), "^ad") ~ "AD",
    stringr::str_detect(tolower(x), "con|ctrl|control") ~ "Control",
    TRUE ~ x
  )
}

as_bool <- function(x) {
  if (is.logical(x)) return(x)
  tolower(as.character(x)) %in% c("true", "t", "1", "yes", "y")
}

merge_cortex_region <- function(x) {
  region_map <- c(
    L1 = "L1",
    L2 = "L23", L23 = "L23", L3 = "L23",
    L4 = "L456", L45 = "L456", L456 = "L456", L5 = "L456", L6 = "L456",
    WM = "WM"
  )

  y <- as.character(x)
  idx <- y %in% names(region_map)
  y[idx] <- unname(region_map[y[idx]])
  y
}

to_meta_df <- function(obj, tissue_name) {
  df <- if (inherits(obj, "Seurat")) obj@meta.data else as.data.frame(obj)
  df <- repair_colnames(df)

  need_cols <- c("group", "sample_id", "area_m", "subcelltype", "unit_id", "dist")
  miss_cols <- setdiff(need_cols, colnames(df))
  if (length(miss_cols) > 0) {
    stop("Missing columns in metadata: ", paste(miss_cols, collapse = ", "))
  }

  df <- df %>%
    dplyr::mutate(
      tissue = tissue_name,
      group = std_group(group),
      sample_id = as.character(sample_id),
      area_m = as.character(area_m),
      subcelltype = as.character(subcelltype),
      unit_id = as.character(unit_id),
      dist = as_bool(dist)
    ) %>%
    dplyr::filter(
      group %in% c("AD", "Control"),
      dist == TRUE,
      !is.na(unit_id),
      unit_id != "",
      !is.na(area_m),
      !is.na(subcelltype)
    )

  if (tissue_name == "Cortex") {
    df$area_m <- merge_cortex_region(df$area_m)
  }

  df$unit_uid <- paste(df$tissue, df$sample_id, df$unit_id, sep = "__")
  df
}

calc_mean_count_per_unit <- function(meta_df, prefix) {
  subtypes <- sort(unique(
    meta_df$subcelltype[stringr::str_detect(meta_df$subcelltype, paste0("^", prefix, "_"))]
  ))

  units <- meta_df %>%
    dplyr::distinct(tissue, group, sample_id, area_m, unit_uid)

  unit_counts <- meta_df %>%
    dplyr::filter(subcelltype %in% subtypes) %>%
    dplyr::count(tissue, group, sample_id, area_m, unit_uid, subcelltype, name = "count") %>%
    dplyr::mutate(subtype = subcelltype) %>%
    dplyr::select(-subcelltype)

  unit_grid <- tidyr::crossing(units, subtype = subtypes)

  unit_counts_full <- unit_grid %>%
    dplyr::left_join(
      unit_counts,
      by = c("tissue", "group", "sample_id", "area_m", "unit_uid", "subtype")
    ) %>%
    dplyr::mutate(count = tidyr::replace_na(count, 0))

  sample_mean <- unit_counts_full %>%
    dplyr::group_by(tissue, sample_id, area_m, group, subtype) %>%
    dplyr::summarise(
      mean_count_per_sample = mean(count, na.rm = TRUE),
      n_units_per_sample = dplyr::n_distinct(unit_uid),
      .groups = "drop"
    )

  group_mean <- sample_mean %>%
    dplyr::group_by(tissue, area_m, group, subtype) %>%
    dplyr::summarise(
      mean_count = mean(mean_count_per_sample, na.rm = TRUE),
      sd_count = stats::sd(mean_count_per_sample, na.rm = TRUE),
      n_samples = dplyr::n_distinct(sample_id),
      sem = sd_count / sqrt(n_samples),
      .groups = "drop"
    )

  list(sample_mean = sample_mean, group_mean = group_mean)
}

p_to_star <- function(p) {
  dplyr::case_when(
    is.na(p) ~ "",
    p < 0.001 ~ "***",
    p < 0.01 ~ "**",
    p < 0.05 ~ "*",
    TRUE ~ ""
  )
}

calc_region_subtype_stats <- function(sample_mean_df, population_name, out_dir) {
  stat_df <- sample_mean_df %>%
    dplyr::group_by(tissue, area_m, subtype) %>%
    dplyr::summarise(
      n_AD = sum(group == "AD"),
      n_Control = sum(group == "Control"),

      mean_AD = mean(mean_count_per_sample[group == "AD"], na.rm = TRUE),
      mean_Control = mean(mean_count_per_sample[group == "Control"], na.rm = TRUE),

      median_AD = median(mean_count_per_sample[group == "AD"], na.rm = TRUE),
      median_Control = median(mean_count_per_sample[group == "Control"], na.rm = TRUE),

      sd_AD = stats::sd(mean_count_per_sample[group == "AD"], na.rm = TRUE),
      sd_Control = stats::sd(mean_count_per_sample[group == "Control"], na.rm = TRUE),

      sem_AD = sd_AD / sqrt(n_AD),
      sem_Control = sd_Control / sqrt(n_Control),

      delta_AD_minus_Control = mean_AD - mean_Control,
      log2FC_AD_vs_Control = log2((mean_AD + 1e-6) / (mean_Control + 1e-6)),

      p_ttest = tryCatch({
        x <- mean_count_per_sample[group == "AD"]
        y <- mean_count_per_sample[group == "Control"]
        if (length(x) >= 2 && length(y) >= 2) {
          stats::t.test(x, y)$p.value
        } else {
          NA_real_
        }
      }, error = function(e) NA_real_),

      p_wilcox = tryCatch({
        x <- mean_count_per_sample[group == "AD"]
        y <- mean_count_per_sample[group == "Control"]
        if (length(x) >= 2 && length(y) >= 2) {
          stats::wilcox.test(x, y, exact = FALSE)$p.value
        } else {
          NA_real_
        }
      }, error = function(e) NA_real_),

      .groups = "drop"
    ) %>%
    dplyr::group_by(tissue) %>%
    dplyr::mutate(
      p_adj_ttest_BH = stats::p.adjust(p_ttest, method = "BH"),
      p_adj_wilcox_BH = stats::p.adjust(p_wilcox, method = "BH")
    ) %>%
    dplyr::ungroup() %>%
    dplyr::mutate(
      population = population_name,
      sig_ttest = p_to_star(p_ttest),
      sig_ttest_BH = p_to_star(p_adj_ttest_BH),
      sig_wilcox = p_to_star(p_wilcox),
      sig_wilcox_BH = p_to_star(p_adj_wilcox_BH)
    ) %>%
    dplyr::select(
      population, tissue, area_m, subtype,
      n_AD, n_Control,
      mean_AD, mean_Control,
      delta_AD_minus_Control,
      log2FC_AD_vs_Control,
      median_AD, median_Control,
      sem_AD, sem_Control,
      p_ttest, p_adj_ttest_BH,
      p_wilcox, p_adj_wilcox_BH,
      sig_ttest, sig_ttest_BH,
      sig_wilcox, sig_wilcox_BH
    )

  write.csv(
    stat_df,
    file.path(out_dir, paste0(population_name, "_region_subtype_AD_vs_Control_stats.csv")),
    row.names = FALSE
  )

  stat_df
}

make_barplot <- function(
  sample_mean_df,
  stat_df,
  population_name,
  tissue_use,
  region_order,
  subtype_order,
  subtype_labels,
  file_prefix,
  width = 12,
  height = 7
) {
  plot_df <- sample_mean_df %>%
    dplyr::filter(
      tissue == tissue_use,
      area_m %in% region_order,
      subtype %in% subtype_order
    ) %>%
    dplyr::mutate(
      area_m = factor(area_m, levels = region_order),
      group = factor(group, levels = c("Control", "AD")),
      subtype = factor(subtype, levels = subtype_order),
      subtype_label = subtype_labels[as.character(subtype)]
    )

  summary_df <- plot_df %>%
    dplyr::group_by(tissue, area_m, subtype, subtype_label, group) %>%
    dplyr::summarise(
      mean_value = mean(mean_count_per_sample, na.rm = TRUE),
      sd_value = stats::sd(mean_count_per_sample, na.rm = TRUE),
      n_sample = dplyr::n_distinct(sample_id),
      sem_value = sd_value / sqrt(n_sample),
      .groups = "drop"
    )

  y_df <- summary_df %>%
    dplyr::group_by(tissue, area_m, subtype, subtype_label) %>%
    dplyr::summarise(
      y_max = max(mean_value + sem_value, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    dplyr::mutate(
      y_max = ifelse(is.finite(y_max), y_max, 0),
      y_pos = ifelse(y_max > 0, y_max * 1.18, 0.08),
      y_tick = ifelse(y_max > 0, y_max * 0.06, 0.03),
      y_text = y_pos + y_tick * 0.9
    )

  sig_df <- stat_df %>%
    dplyr::filter(
      population == population_name,
      tissue == tissue_use,
      area_m %in% region_order,
      subtype %in% subtype_order
    ) %>%
    dplyr::mutate(
      area_m = factor(area_m, levels = region_order),
      subtype = factor(subtype, levels = subtype_order),
      subtype_label = subtype_labels[as.character(subtype)],
      p_plot = .data[[p_for_plot]],
      sig_plot = p_to_star(p_plot)
    ) %>%
    dplyr::filter(!is.na(p_plot), p_plot < 0.05) %>%
    dplyr::left_join(
      y_df,
      by = c("tissue", "area_m", "subtype", "subtype_label")
    )

  p <- ggplot(summary_df, aes(x = group, y = mean_value, fill = group)) +
    geom_col(
      width = 0.58,
      color = "black",
      linewidth = 0.35
    ) +
    geom_errorbar(
      aes(ymin = mean_value - sem_value, ymax = mean_value + sem_value),
      width = 0.18,
      linewidth = 0.35,
      color = "black"
    ) +
    facet_grid(
      rows = vars(subtype_label),
      cols = vars(area_m),
      scales = "free_y"
    ) +
    scale_fill_manual(
      values = c(Control = "#BFE3EA", AD = "#F2B8AE"),
      breaks = c("Control", "AD"),
      name = NULL
    ) +
    scale_y_continuous(
      expand = expansion(mult = c(0, 0.28))
    ) +
    labs(
      title = paste0(population_name, " subtype abundance by brain region"),
      x = NULL,
      y = "Mean cell count per digital NVU"
    ) +
    theme_classic(base_size = 8, base_family = "Arial") +
    theme(
      plot.title = element_text(size = 11, face = "bold", hjust = 0.5),
      axis.line = element_line(linewidth = 0.35, colour = "black"),
      axis.ticks = element_line(linewidth = 0.35, colour = "black"),
      axis.text.x = element_text(size = 7, angle = 0, hjust = 0.5, colour = "black"),
      axis.text.y = element_text(size = 6.5, colour = "black"),
      axis.title.y = element_text(size = 8, colour = "black"),
      strip.background = element_blank(),
      strip.text.x = element_text(size = 8.5, face = "bold", colour = "black"),
      strip.text.y = element_text(size = 7.5, face = "bold", colour = "black", angle = 0),
      panel.spacing = unit(1.0, "lines"),
      legend.position = "top",
      legend.text = element_text(size = 7),
      plot.margin = margin(5, 5, 5, 5)
    )

  if (show_points) {
    p <- p +
      geom_point(
        data = plot_df,
        aes(x = group, y = mean_count_per_sample, colour = group),
        inherit.aes = FALSE,
        position = position_jitter(width = 0.09, height = 0),
        size = 0.8,
        alpha = 0.75
      ) +
      scale_colour_manual(
        values = c(Control = "#5DBCD2", AD = "#E85C4A"),
        guide = "none"
      )
  }

  if (nrow(sig_df) > 0) {
    p <- p +
      geom_segment(
        data = sig_df,
        aes(x = 1, xend = 2, y = y_pos, yend = y_pos),
        inherit.aes = FALSE,
        linewidth = 0.35,
        colour = "black"
      ) +
      geom_segment(
        data = sig_df,
        aes(x = 1, xend = 1, y = y_pos - y_tick, yend = y_pos),
        inherit.aes = FALSE,
        linewidth = 0.35,
        colour = "black"
      ) +
      geom_segment(
        data = sig_df,
        aes(x = 2, xend = 2, y = y_pos - y_tick, yend = y_pos),
        inherit.aes = FALSE,
        linewidth = 0.35,
        colour = "black"
      ) +
      geom_text(
        data = sig_df,
        aes(x = 1.5, y = y_text, label = sig_plot),
        inherit.aes = FALSE,
        size = 3,
        fontface = "bold",
        colour = "black"
      )
  }

  ggsave(
    filename = file.path(out_dir, paste0(file_prefix, ".pdf")),
    plot = p,
    width = width,
    height = height,
    device = cairo_pdf
  )

  ggsave(
    filename = file.path(out_dir, paste0(file_prefix, ".png")),
    plot = p,
    width = width,
    height = height,
    dpi = 600,
    bg = "white"
  )

  p
}

# 3. Prepare metadata

NVU_hip_meta <- to_meta_df(NVU_hip, "Hippocampus")
NVU_cortex_meta <- to_meta_df(NVU_cortex, "Cortex")

meta_all <- dplyr::bind_rows(NVU_hip_meta, NVU_cortex_meta)

# 4. Calculate subtype mean count per sample

astro_res <- calc_mean_count_per_unit(meta_all, "Astro")
micro_res <- calc_mean_count_per_unit(meta_all, "Micro")

write.csv(
  astro_res$sample_mean,
  file.path(out_dir, "Astro_sample_mean_count_per_unit.csv"),
  row.names = FALSE
)

write.csv(
  micro_res$sample_mean,
  file.path(out_dir, "Micro_sample_mean_count_per_unit.csv"),
  row.names = FALSE
)

write.csv(
  astro_res$group_mean,
  file.path(out_dir, "Astro_group_mean_count_per_unit.csv"),
  row.names = FALSE
)

write.csv(
  micro_res$group_mean,
  file.path(out_dir, "Micro_group_mean_count_per_unit.csv"),
  row.names = FALSE
)

# 5. Statistics

astro_stats <- calc_region_subtype_stats(
  astro_res$sample_mean,
  "Astro",
  out_dir
)

micro_stats <- calc_region_subtype_stats(
  micro_res$sample_mean,
  "Micro",
  out_dir
)

all_subtype_stats <- dplyr::bind_rows(astro_stats, micro_stats)

write.csv(
  all_subtype_stats,
  file.path(out_dir, "Astro_Micro_region_subtype_AD_vs_Control_stats.csv"),
  row.names = FALSE
)

sig_results <- all_subtype_stats %>%
  dplyr::filter(!is.na(.data[[p_for_plot]]), .data[[p_for_plot]] < 0.05) %>%
  dplyr::arrange(population, tissue, area_m, .data[[p_for_plot]])

write.csv(
  sig_results,
  file.path(out_dir, "Significant_Astro_Micro_region_subtype_AD_vs_Control.csv"),
  row.names = FALSE
)

# 6. Plot

hip_regions <- c("CA1", "CA2", "CA3", "CA4", "DG", "FAS", "SLRM")
cortex_regions <- c("L1", "L23", "L456", "WM")

astro_order <- paste0("Astro_", 0:4)
micro_order <- paste0("Micro_", 0:4)

astro_labels <- c(
  Astro_0 = "Astro.0",
  Astro_1 = "Astro.1",
  Astro_2 = "DAA.1",
  Astro_3 = "DAA.2",
  Astro_4 = "Astro.4"
)

micro_labels <- c(
  Micro_0 = "Micro.0",
  Micro_1 = "Micro.1",
  Micro_2 = "DAM",
  Micro_3 = "Micro.3",
  Micro_4 = "Micro.4"
)

p_astro_hip <- make_barplot(
  sample_mean_df = astro_res$sample_mean,
  stat_df = astro_stats,
  population_name = "Astro",
  tissue_use = "Hippocampus",
  region_order = hip_regions,
  subtype_order = astro_order,
  subtype_labels = astro_labels,
  file_prefix = "Astro_Hippocampus_AD_vs_Control_barplot",
  width = 13,
  height = 7.5
)

p_astro_cortex <- make_barplot(
  sample_mean_df = astro_res$sample_mean,
  stat_df = astro_stats,
  population_name = "Astro",
  tissue_use = "Cortex",
  region_order = cortex_regions,
  subtype_order = astro_order,
  subtype_labels = astro_labels,
  file_prefix = "Astro_Cortex_AD_vs_Control_barplot",
  width = 9,
  height = 7.5
)

p_micro_hip <- make_barplot(
  sample_mean_df = micro_res$sample_mean,
  stat_df = micro_stats,
  population_name = "Micro",
  tissue_use = "Hippocampus",
  region_order = hip_regions,
  subtype_order = micro_order,
  subtype_labels = micro_labels,
  file_prefix = "Micro_Hippocampus_AD_vs_Control_barplot",
  width = 13,
  height = 7.5
)

p_micro_cortex <- make_barplot(
  sample_mean_df = micro_res$sample_mean,
  stat_df = micro_stats,
  population_name = "Micro",
  tissue_use = "Cortex",
  region_order = cortex_regions,
  subtype_order = micro_order,
  subtype_labels = micro_labels,
  file_prefix = "Micro_Cortex_AD_vs_Control_barplot",
  width = 9,
  height = 7.5
)

print(p_astro_hip)
print(p_astro_cortex)
print(p_micro_hip)
print(p_micro_cortex)

cat("Done.\nOutput directory:\n", out_dir, "\n")

In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
  library(scales)
})

# 0. Paths and settings

out_dir <- "../results/08_DAA_DAM/DAA_DAM_density_selected_regions_no_outlier"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

rds_hip <- "../results/05_nvu_analysis_results/Hip/200/Hip_merged-subtype_0330.rds"
rds_cortex <- "../results/05_nvu_analysis_results/Cortex/200/Cortex_merged-subtype_0330.rds"

data_dir <- "../data"

hip_area_candidates <- c(
  file.path(data_dir, "Hip_region_bin50_summary.csv"),
  file.path(data_dir, "Hippocampus_region_bin50_summary.csv")
)

cortex_area_candidates <- c(
  file.path(data_dir, "Cortex_region_bin50_summary.csv"),
  file.path(data_dir, "PFC_region_bin50_summary.csv"),
  file.path(data_dir, "Prefrontal_region_bin50_summary.csv")
)

selected_regions <- c("CA1", "FAS", "SLRM", "L23", "L456")

show_points <- FALSE
remove_outliers <- TRUE
outlier_k <- 1.5
p_for_plot <- "p_adj_ttest_BH"

# 1. Read data

NVU_hip <- readRDS(rds_hip)
NVU_cortex <- readRDS(rds_cortex)

# 2. Helper functions

repair_colnames <- function(df) {
  nm <- names(df)
  bad <- is.na(nm) | nm == ""
  nm[bad] <- paste0("blank_col_", seq_len(sum(bad)))
  names(df) <- make.unique(nm, sep = "_")
  df
}

std_group <- function(x) {
  x <- as.character(x)
  dplyr::case_when(
    stringr::str_detect(tolower(x), "^ad") ~ "AD",
    stringr::str_detect(tolower(x), "con|ctrl|control") ~ "Control",
    TRUE ~ x
  )
}

as_bool <- function(x) {
  if (is.logical(x)) return(x)
  tolower(as.character(x)) %in% c("true", "t", "1", "yes", "y")
}

merge_cortex_region <- function(x) {
  region_map <- c(
    L1 = "L1",
    L2 = "L23", L23 = "L23", L3 = "L23",
    L4 = "L456", L45 = "L456", L456 = "L456",
    L5 = "L456", L6 = "L456",
    WM = "WM",
    WhiteMatter = "WM", white_matter = "WM"
  )

  y <- as.character(x)
  y <- stringr::str_replace_all(y, "^Layer", "L")
  y <- stringr::str_replace_all(y, "^layer", "L")
  y <- stringr::str_replace_all(y, " ", "")
  idx <- y %in% names(region_map)
  y[idx] <- unname(region_map[y[idx]])
  y
}

find_existing_file <- function(candidates, pattern = NULL, search_dir = NULL) {
  hit <- candidates[file.exists(candidates)]
  if (length(hit) > 0) return(hit[1])

  if (!is.null(pattern) && !is.null(search_dir) && dir.exists(search_dir)) {
    auto_hit <- list.files(search_dir, pattern = pattern, full.names = TRUE, ignore.case = TRUE)
    if (length(auto_hit) > 0) return(auto_hit[1])
  }

  stop("Cannot find area summary file. Tried:\n", paste(candidates, collapse = "\n"))
}

to_meta_df <- function(obj, tissue_name) {
  df <- if (inherits(obj, "Seurat")) obj@meta.data else as.data.frame(obj)
  df <- repair_colnames(df)

  need_cols <- c("group", "sample_id", "area_m", "subcelltype", "unit_id", "dist")
  miss_cols <- setdiff(need_cols, colnames(df))
  if (length(miss_cols) > 0) {
    stop("Missing columns in metadata: ", paste(miss_cols, collapse = ", "))
  }

  df <- df %>%
    dplyr::mutate(
      tissue = tissue_name,
      group = std_group(group),
      sample_id = as.character(sample_id),
      area_m = as.character(area_m),
      subcelltype = as.character(subcelltype),
      unit_id = as.character(unit_id),
      dist = as_bool(dist)
    ) %>%
    dplyr::filter(
      group %in% c("AD", "Control"),
      dist == TRUE,
      !is.na(sample_id),
      !is.na(area_m),
      !is.na(subcelltype),
      !is.na(unit_id),
      unit_id != ""
    )

  if (tissue_name == "Cortex") {
    df$area_m <- merge_cortex_region(df$area_m)
  }

  df$unit_uid <- paste(df$tissue, df$sample_id, df$unit_id, sep = "__")
  df
}

pick_sample_col <- function(area_df, meta_df) {
  candidate_cols <- intersect(
    c("sample_id", "sample", "chip", "orig.ident", "Sample", "sample_name", "slide", "Slide"),
    colnames(area_df)
  )

  if (length(candidate_cols) == 0) {
    candidate_cols <- colnames(area_df)[sapply(area_df, function(x) is.character(x) || is.factor(x))]
  }

  scores <- sapply(candidate_cols, function(cc) {
    length(intersect(unique(as.character(area_df[[cc]])), unique(as.character(meta_df$sample_id))))
  })

  best_col <- candidate_cols[which.max(scores)]
  message("Area file sample column selected: ", best_col, " ; matched samples: ", max(scores))
  best_col
}

pick_area_col <- function(area_df, tissue_name, meta_df) {
  preferred_cols <- c(
    "area_m", "area", "region", "Region", "brain_region", "Brain_region",
    "brain_area", "Brain_area", "area_name", "Area", "variable", "Variable",
    "annotation", "region_name", "layer", "Layer", "subregion", "Subregion"
  )

  candidate_cols <- intersect(preferred_cols, colnames(area_df))

  if (length(candidate_cols) == 0) {
    candidate_cols <- colnames(area_df)[sapply(area_df, function(x) is.character(x) || is.factor(x))]
  }

  meta_regions <- unique(as.character(meta_df$area_m))
  if (tissue_name == "Cortex") {
    meta_regions <- unique(merge_cortex_region(meta_regions))
  }

  scores <- sapply(candidate_cols, function(cc) {
    vals <- as.character(area_df[[cc]])
    vals2 <- if (tissue_name == "Cortex") merge_cortex_region(vals) else vals
    length(intersect(unique(vals2), meta_regions))
  })

  best_col <- candidate_cols[which.max(scores)]
  message("Area file region column selected: ", best_col, " ; matched regions: ", max(scores))
  best_col
}

read_area_summary <- function(area_file, tissue_name, meta_df) {
  area_df <- read.csv(area_file, check.names = FALSE)
  area_df <- repair_colnames(area_df)

  message("\nReading area file: ", area_file)
  message("Columns: ", paste(colnames(area_df), collapse = ", "))

  if (!"area_mm2" %in% colnames(area_df)) {
    area_mm2_candidates <- c("area_mm2", "Area_mm2", "area.mm2")
    area_mm2_col <- intersect(area_mm2_candidates, colnames(area_df))
    numeric_cols <- colnames(area_df)[sapply(area_df, is.numeric)]
    numeric_cols <- setdiff(numeric_cols, c("X", "X.1"))

    if (length(area_mm2_col) > 0) {
      area_df$area_mm2 <- as.numeric(area_df[[area_mm2_col[1]]])
    } else if (length(numeric_cols) > 0) {
      area_df$area_mm2 <- as.numeric(area_df[[numeric_cols[1]]])
      message("area_mm2 selected by numeric fallback: ", numeric_cols[1])
    } else {
      stop("No numeric area column found in area file: ", area_file)
    }
  }

  sample_col <- pick_sample_col(area_df, meta_df)
  area_col <- pick_area_col(area_df, tissue_name, meta_df)

  sample_group <- meta_df %>%
    dplyr::distinct(sample_id, group)

  area_use <- area_df %>%
    dplyr::mutate(
      tissue = tissue_name,
      sample_id = as.character(.data[[sample_col]]),
      area_m = as.character(.data[[area_col]]),
      area_mm2 = as.numeric(area_mm2)
    ) %>%
    dplyr::filter(
      !is.na(sample_id),
      !is.na(area_m),
      !is.na(area_mm2),
      area_mm2 > 0
    )

  if (tissue_name == "Cortex") {
    area_use$area_m <- merge_cortex_region(area_use$area_m)
  }

  area_use <- area_use %>%
    dplyr::filter(area_m %in% unique(meta_df$area_m)) %>%
    dplyr::distinct(tissue, sample_id, area_m, area_mm2) %>%
    dplyr::group_by(tissue, sample_id, area_m) %>%
    dplyr::summarise(
      area_mm2 = sum(unique(area_mm2), na.rm = TRUE),
      .groups = "drop"
    ) %>%
    dplyr::left_join(sample_group, by = "sample_id") %>%
    dplyr::filter(group %in% c("AD", "Control"))

  if (nrow(area_use) == 0) {
    stop("No valid matched area rows remained for ", tissue_name)
  }

  area_use
}

label_DAA_DAM_detail <- function(x) {
  x <- as.character(x)
  dplyr::case_when(
    x %in% c("Astro_2", "DAA.1", "DAA_1", "DAA1") ~ "DAA.1",
    x %in% c("Astro_3", "DAA.2", "DAA_2", "DAA2") ~ "DAA.2",
    x %in% c("Micro_2", "DAM", "DAM_1") ~ "DAM",
    TRUE ~ NA_character_
  )
}

label_DAA_DAM_collapsed <- function(x) {
  x <- as.character(x)
  dplyr::case_when(
    x %in% c("Astro_2", "Astro_3", "DAA.1", "DAA.2", "DAA_1", "DAA_2", "DAA1", "DAA2") ~ "DAA",
    x %in% c("Micro_2", "DAM", "DAM_1") ~ "DAM",
    TRUE ~ NA_character_
  )
}

p_to_star <- function(p) {
  dplyr::case_when(
    is.na(p) ~ "",
    p < 0.001 ~ "***",
    p < 0.01 ~ "**",
    p < 0.05 ~ "*",
    TRUE ~ ""
  )
}

make_density_table <- function(meta_df, area_df, mode = c("detail", "collapsed")) {
  mode <- match.arg(mode)

  subtype_levels <- if (mode == "detail") {
    c("DAA.1", "DAA.2", "DAM")
  } else {
    c("DAA", "DAM")
  }

  meta_labeled <- meta_df %>%
    dplyr::mutate(
      subtype_focus = if (mode == "detail") {
        label_DAA_DAM_detail(subcelltype)
      } else {
        label_DAA_DAM_collapsed(subcelltype)
      }
    ) %>%
    dplyr::filter(!is.na(subtype_focus))

  sample_area <- meta_df %>%
    dplyr::distinct(tissue, group, sample_id, area_m) %>%
    dplyr::left_join(
      area_df %>% dplyr::select(tissue, group, sample_id, area_m, area_mm2),
      by = c("tissue", "group", "sample_id", "area_m")
    ) %>%
    dplyr::filter(!is.na(area_mm2), area_mm2 > 0)

  sample_grid <- sample_area[rep(seq_len(nrow(sample_area)), each = length(subtype_levels)), , drop = FALSE]
  sample_grid$subtype_focus <- rep(subtype_levels, times = nrow(sample_area))

  count_df <- meta_labeled %>%
    dplyr::count(tissue, group, sample_id, area_m, subtype_focus, name = "cell_count")

  density_df <- sample_grid %>%
    dplyr::left_join(
      count_df,
      by = c("tissue", "group", "sample_id", "area_m", "subtype_focus")
    ) %>%
    dplyr::mutate(
      cell_count = tidyr::replace_na(cell_count, 0L),
      density_cells_mm2 = cell_count / area_mm2,
      mode = mode
    )

  density_df
}

flag_outliers_iqr <- function(df, value_col = "density_cells_mm2", k = 1.5) {
  df %>%
    dplyr::group_by(mode, tissue, area_m, subtype_focus, group) %>%
    dplyr::mutate(
      q1 = stats::quantile(.data[[value_col]], 0.25, na.rm = TRUE),
      q3 = stats::quantile(.data[[value_col]], 0.75, na.rm = TRUE),
      iqr = q3 - q1,
      lower = q1 - k * iqr,
      upper = q3 + k * iqr,
      is_outlier = ifelse(
        is.na(iqr) | iqr == 0,
        FALSE,
        .data[[value_col]] < lower | .data[[value_col]] > upper
      )
    ) %>%
    dplyr::ungroup()
}

calc_density_stats <- function(density_df, label_name) {
  stat_df <- density_df %>%
    dplyr::group_by(mode, tissue, area_m, subtype_focus) %>%
    dplyr::summarise(
      n_AD = sum(group == "AD"),
      n_Control = sum(group == "Control"),

      mean_AD = mean(density_cells_mm2[group == "AD"], na.rm = TRUE),
      mean_Control = mean(density_cells_mm2[group == "Control"], na.rm = TRUE),

      median_AD = median(density_cells_mm2[group == "AD"], na.rm = TRUE),
      median_Control = median(density_cells_mm2[group == "Control"], na.rm = TRUE),

      sd_AD = stats::sd(density_cells_mm2[group == "AD"], na.rm = TRUE),
      sd_Control = stats::sd(density_cells_mm2[group == "Control"], na.rm = TRUE),

      sem_AD = sd_AD / sqrt(n_AD),
      sem_Control = sd_Control / sqrt(n_Control),

      delta_AD_minus_Control = mean_AD - mean_Control,
      log2FC_AD_vs_Control = log2((mean_AD + 1e-6) / (mean_Control + 1e-6)),

      p_ttest = tryCatch({
        x <- density_cells_mm2[group == "AD"]
        y <- density_cells_mm2[group == "Control"]
        if (length(x) >= 2 && length(y) >= 2) stats::t.test(x, y)$p.value else NA_real_
      }, error = function(e) NA_real_),

      p_wilcox = tryCatch({
        x <- density_cells_mm2[group == "AD"]
        y <- density_cells_mm2[group == "Control"]
        if (length(x) >= 2 && length(y) >= 2) stats::wilcox.test(x, y, exact = FALSE)$p.value else NA_real_
      }, error = function(e) NA_real_),

      .groups = "drop"
    ) %>%
    dplyr::group_by(mode, tissue) %>%
    dplyr::mutate(
      p_adj_ttest_BH = stats::p.adjust(p_ttest, method = "BH"),
      p_adj_wilcox_BH = stats::p.adjust(p_wilcox, method = "BH")
    ) %>%
    dplyr::ungroup() %>%
    dplyr::mutate(
      analysis = label_name,
      sig_ttest = p_to_star(p_ttest),
      sig_ttest_BH = p_to_star(p_adj_ttest_BH),
      sig_wilcox = p_to_star(p_wilcox),
      sig_wilcox_BH = p_to_star(p_adj_wilcox_BH)
    )

  stat_df
}

make_selected_density_barplot <- function(
  density_df,
  stat_df,
  title_text,
  file_prefix,
  width = 10,
  height = 4.8
) {
  plot_df <- density_df %>%
    dplyr::filter(area_m %in% selected_regions) %>%
    dplyr::mutate(
      area_m = factor(area_m, levels = selected_regions),
      group = factor(group, levels = c("Control", "AD")),
      subtype_focus = factor(subtype_focus, levels = c("DAA", "DAM", "DAA.1", "DAA.2"))
    )

  summary_df <- plot_df %>%
    dplyr::group_by(tissue, area_m, subtype_focus, group) %>%
    dplyr::summarise(
      mean_density = mean(density_cells_mm2, na.rm = TRUE),
      sd_density = stats::sd(density_cells_mm2, na.rm = TRUE),
      n_sample = dplyr::n_distinct(sample_id),
      sem_density = sd_density / sqrt(n_sample),
      .groups = "drop"
    )

  y_df <- summary_df %>%
    dplyr::group_by(tissue, area_m, subtype_focus) %>%
    dplyr::summarise(
      y_max = max(mean_density + sem_density, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    dplyr::mutate(
      y_max = ifelse(is.finite(y_max), y_max, 0),
      y_pos = ifelse(y_max > 0, y_max * 1.18, 0.08),
      y_tick = ifelse(y_max > 0, y_max * 0.06, 0.03),
      y_text = y_pos + y_tick * 0.9
    )

  sig_df <- stat_df %>%
    dplyr::filter(area_m %in% selected_regions) %>%
    dplyr::mutate(
      area_m = factor(area_m, levels = selected_regions),
      subtype_focus = factor(subtype_focus, levels = levels(plot_df$subtype_focus)),
      p_plot = .data[[p_for_plot]],
      sig_plot = p_to_star(p_plot)
    ) %>%
    dplyr::filter(!is.na(p_plot), p_plot < 0.05) %>%
    dplyr::left_join(
      y_df,
      by = c("tissue", "area_m", "subtype_focus")
    )

  p <- ggplot(summary_df, aes(x = group, y = mean_density, fill = group)) +
    geom_col(width = 0.58, color = "black", linewidth = 0.35) +
    geom_errorbar(
      aes(ymin = mean_density - sem_density, ymax = mean_density + sem_density),
      width = 0.18,
      linewidth = 0.35,
      color = "black"
    ) +
    facet_grid(
      rows = vars(subtype_focus),
      cols = vars(area_m),
      scales = "free_y"
    ) +
    scale_fill_manual(
      values = c(Control = "#BFE3EA", AD = "#F2B8AE"),
      breaks = c("Control", "AD"),
      name = NULL
    ) +
    scale_y_continuous(expand = expansion(mult = c(0, 0.30))) +
    labs(
      title = title_text,
      x = NULL,
      y = expression("Density (cells mm"^-2*")")
    ) +
    theme_classic(base_size = 8, base_family = "Arial") +
    theme(
      plot.title = element_text(size = 11, face = "bold", hjust = 0.5),
      axis.line = element_line(linewidth = 0.35, colour = "black"),
      axis.ticks = element_line(linewidth = 0.35, colour = "black"),
      axis.text.x = element_text(size = 7, colour = "black"),
      axis.text.y = element_text(size = 6.5, colour = "black"),
      axis.title.y = element_text(size = 8, colour = "black"),
      strip.background = element_blank(),
      strip.text.x = element_text(size = 8.5, face = "bold", colour = "black"),
      strip.text.y = element_text(size = 7.5, face = "bold", colour = "black", angle = 0),
      panel.spacing = unit(0.95, "lines"),
      legend.position = "top",
      legend.text = element_text(size = 7),
      plot.margin = margin(5, 5, 5, 5)
    )

  if (show_points) {
    p <- p +
      geom_point(
        data = plot_df,
        aes(x = group, y = density_cells_mm2, colour = group),
        inherit.aes = FALSE,
        position = position_jitter(width = 0.08, height = 0),
        size = 0.8,
        alpha = 0.7
      ) +
      scale_colour_manual(values = c(Control = "#5DBCD2", AD = "#E85C4A"), guide = "none")
  }

  if (nrow(sig_df) > 0) {
    p <- p +
      geom_segment(
        data = sig_df,
        aes(x = 1, xend = 2, y = y_pos, yend = y_pos),
        inherit.aes = FALSE,
        linewidth = 0.35
      ) +
      geom_segment(
        data = sig_df,
        aes(x = 1, xend = 1, y = y_pos - y_tick, yend = y_pos),
        inherit.aes = FALSE,
        linewidth = 0.35
      ) +
      geom_segment(
        data = sig_df,
        aes(x = 2, xend = 2, y = y_pos - y_tick, yend = y_pos),
        inherit.aes = FALSE,
        linewidth = 0.35
      ) +
      geom_text(
        data = sig_df,
        aes(x = 1.5, y = y_text, label = sig_plot),
        inherit.aes = FALSE,
        size = 3,
        fontface = "bold"
      )
  }

  ggsave(file.path(out_dir, paste0(file_prefix, ".pdf")), p, width = width, height = height, device = cairo_pdf)
  ggsave(file.path(out_dir, paste0(file_prefix, ".png")), p, width = width, height = height, dpi = 600, bg = "white")

  p
}

# 3. Metadata and area

NVU_hip_meta <- to_meta_df(NVU_hip, "Hippocampus")
NVU_cortex_meta <- to_meta_df(NVU_cortex, "Cortex")
meta_all <- dplyr::bind_rows(NVU_hip_meta, NVU_cortex_meta)

hip_area_file <- find_existing_file(
  hip_area_candidates,
  pattern = "Hip.*region.*bin50.*summary.*csv|Hippocampus.*region.*bin50.*summary.*csv",
  search_dir = data_dir
)

cortex_area_file <- find_existing_file(
  cortex_area_candidates,
  pattern = "Cortex.*region.*bin50.*summary.*csv|PFC.*region.*bin50.*summary.*csv|Prefrontal.*region.*bin50.*summary.*csv",
  search_dir = data_dir
)

hip_area <- read_area_summary(hip_area_file, "Hippocampus", NVU_hip_meta)
cortex_area <- read_area_summary(cortex_area_file, "Cortex", NVU_cortex_meta)
area_all <- dplyr::bind_rows(hip_area, cortex_area)

write.csv(area_all, file.path(out_dir, "region_area_mm2_used_for_density.csv"), row.names = FALSE)

# 4. Density tables

density_detail_raw <- make_density_table(meta_all, area_all, mode = "detail")
density_collapsed_raw <- make_density_table(meta_all, area_all, mode = "collapsed")

density_detail_flagged <- flag_outliers_iqr(density_detail_raw, k = outlier_k)
density_collapsed_flagged <- flag_outliers_iqr(density_collapsed_raw, k = outlier_k)

write.csv(density_detail_flagged, file.path(out_dir, "DAA1_DAA2_DAM_density_with_outlier_flag.csv"), row.names = FALSE)
write.csv(density_collapsed_flagged, file.path(out_dir, "DAA_DAM_density_with_outlier_flag.csv"), row.names = FALSE)

outliers_all <- dplyr::bind_rows(density_detail_flagged, density_collapsed_flagged) %>%
  dplyr::filter(is_outlier)

write.csv(outliers_all, file.path(out_dir, "Removed_outliers_IQR_density.csv"), row.names = FALSE)

if (remove_outliers) {
  density_detail <- density_detail_flagged %>% dplyr::filter(!is_outlier)
  density_collapsed <- density_collapsed_flagged %>% dplyr::filter(!is_outlier)
} else {
  density_detail <- density_detail_flagged
  density_collapsed <- density_collapsed_flagged
}

write.csv(density_detail, file.path(out_dir, "DAA1_DAA2_DAM_density_no_outlier_used.csv"), row.names = FALSE)
write.csv(density_collapsed, file.path(out_dir, "DAA_DAM_density_no_outlier_used.csv"), row.names = FALSE)

# 5. Statistics before and after outlier removal

stats_detail_raw <- calc_density_stats(density_detail_raw, "raw_detail")
stats_collapsed_raw <- calc_density_stats(density_collapsed_raw, "raw_collapsed")

stats_detail_no_outlier <- calc_density_stats(density_detail, "no_outlier_detail")
stats_collapsed_no_outlier <- calc_density_stats(density_collapsed, "no_outlier_collapsed")

all_stats <- dplyr::bind_rows(
  stats_detail_raw,
  stats_collapsed_raw,
  stats_detail_no_outlier,
  stats_collapsed_no_outlier
)

write.csv(
  all_stats,
  file.path(out_dir, "DAA_DAM_density_stats_raw_and_no_outlier.csv"),
  row.names = FALSE
)

stats_selected_no_outlier <- all_stats %>%
  dplyr::filter(
    analysis %in% c("no_outlier_detail", "no_outlier_collapsed"),
    area_m %in% selected_regions
  ) %>%
  dplyr::arrange(mode, tissue, area_m, subtype_focus)

write.csv(
  stats_selected_no_outlier,
  file.path(out_dir, "Selected_regions_DAA_DAM_density_stats_no_outlier.csv"),
  row.names = FALSE
)

# 6. Diagnostic: why collapsed can become significant

diagnostic_compare <- stats_selected_no_outlier %>%
  dplyr::select(
    analysis, mode, tissue, area_m, subtype_focus,
    n_AD, n_Control,
    mean_AD, mean_Control,
    delta_AD_minus_Control,
    sd_AD, sd_Control,
    p_ttest, p_adj_ttest_BH,
    sig_ttest_BH
  ) %>%
  dplyr::arrange(tissue, area_m, mode, subtype_focus)

write.csv(
  diagnostic_compare,
  file.path(out_dir, "Diagnostic_detail_vs_collapsed_significance_no_outlier.csv"),
  row.names = FALSE
)

# 7. Plot selected regions in one figure

p_collapsed_selected <- make_selected_density_barplot(
  density_df = density_collapsed,
  stat_df = stats_collapsed_no_outlier,
  title_text = "DAA and DAM density in selected vulnerable digital NVU regions",
  file_prefix = "Selected_regions_DAA_DAM_density_no_outlier",
  width = 9.5,
  height = 4.4
)

p_detail_selected <- make_selected_density_barplot(
  density_df = density_detail,
  stat_df = stats_detail_no_outlier,
  title_text = "DAA.1, DAA.2 and DAM density in selected vulnerable digital NVU regions",
  file_prefix = "Selected_regions_DAA1_DAA2_DAM_density_no_outlier",
  width = 9.5,
  height = 5.4
)

print(p_collapsed_selected)
print(p_detail_selected)

cat("Done.\nOutput directory:\n", out_dir, "\n")

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
  library(scales)
})

# 0. Paths and settings

in_dir <- "../results/08_DAA_DAM/DAA_DAM_density_selected_regions_no_outlier"
out_dir <- "../results/08_DAA_DAM/DAA_DAM_density_selected_regions_nature_style"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

stats_file <- file.path(in_dir, "DAA_DAM_density_stats_raw_and_no_outlier.csv")

selected_regions <- c("L23", "L456", "CA1", "FAS", "SLRM")
subtype_order <- c("DAA.1", "DAA.2", "DAM")

p_for_plot <- "p_adj_ttest_BH"

region_colors <- c(
  L23  = "#4E79A7",
  L456 = "#76B7B2",
  CA1  = "#E15759",
  FAS  = "#F28E2B",
  SLRM = "#59A14F"
)

# 1. Read statistics

stats_df <- read.csv(stats_file, check.names = FALSE)

need_cols <- c(
  "analysis", "area_m", "subtype_focus",
  "log2FC_AD_vs_Control", p_for_plot
)

miss_cols <- setdiff(need_cols, colnames(stats_df))
if (length(miss_cols) > 0) {
  stop("stats file missing columns: ", paste(miss_cols, collapse = ", "))
}

plot_df <- stats_df %>%
  dplyr::filter(
    analysis == "no_outlier_detail",
    area_m %in% selected_regions,
    subtype_focus %in% subtype_order
  ) %>%
  dplyr::mutate(
    area_m = factor(area_m, levels = selected_regions),
    subtype_focus = factor(subtype_focus, levels = subtype_order),
    p_plot = .data[[p_for_plot]],
    sig_plot = dplyr::case_when(
      is.na(p_plot) ~ "",
      p_plot < 0.001 ~ "***",
      p_plot < 0.01 ~ "**",
      p_plot < 0.05 ~ "*",
      TRUE ~ "ns"
    ),
    log2FC_plot = log2FC_AD_vs_Control,
    minus_log10_p = -log10(p_plot)
  ) %>%
  dplyr::arrange(area_m, subtype_focus) %>%
  dplyr::group_by(area_m) %>%
  dplyr::mutate(
    subtype_index = as.integer(subtype_focus),
    region_index = as.integer(area_m),
    x_pos = (region_index - 1) * 4 + subtype_index
  ) %>%
  dplyr::ungroup() %>%
  dplyr::mutate(
    label_y = ifelse(log2FC_plot >= 0, log2FC_plot + 0.18, log2FC_plot - 0.18),
    label_vjust = ifelse(log2FC_plot >= 0, 0, 1)
  )

write.csv(
  plot_df,
  file.path(out_dir, "Source_stats_for_top_region_bar_lollipop.csv"),
  row.names = FALSE
)

# 2. Top brain-region bar data

region_bar_df <- plot_df %>%
  dplyr::group_by(area_m, region_index) %>%
  dplyr::summarise(
    xmin = min(x_pos) - 0.45,
    xmax = max(x_pos) + 0.45,
    xmid = mean(c(xmin, xmax)),
    .groups = "drop"
  )

# 3. Theme

theme_nature <- function(base_size = 8) {
  theme_classic(base_size = base_size, base_family = "Arial") +
    theme(
      axis.line = element_line(linewidth = 0.35, colour = "black"),
      axis.ticks = element_line(linewidth = 0.35, colour = "black"),
      axis.text = element_text(colour = "black"),
      axis.title = element_text(colour = "black"),
      legend.title = element_text(size = base_size),
      legend.text = element_text(size = base_size),
      plot.title = element_text(face = "bold", hjust = 0.5, size = base_size + 4),
      plot.subtitle = element_text(hjust = 0.5, size = base_size, colour = "grey35"),
      panel.grid = element_blank(),
      plot.margin = margin(8, 8, 8, 8)
    )
}

# 4. Plot

y_lim <- max(abs(plot_df$log2FC_plot), na.rm = TRUE)
if (!is.finite(y_lim)) y_lim <- 1
y_lim <- ceiling((y_lim + 0.8) * 2) / 2
top_ymin <- y_lim + 0.18
top_ymax <- y_lim + 0.42
top_label_y <- y_lim + 0.30

p_top_region_lollipop <- ggplot(plot_df, aes(x = x_pos, y = log2FC_plot)) +
  geom_hline(yintercept = 0, linewidth = 0.35, colour = "grey45") +

  geom_segment(
    aes(
      x = x_pos,
      xend = x_pos,
      y = 0,
      yend = log2FC_plot,
      colour = area_m
    ),
    linewidth = 0.8,
    alpha = 0.88
  ) +

  geom_point(
    aes(fill = area_m, size = minus_log10_p),
    shape = 21,
    colour = "black",
    stroke = 0.30,
    alpha = 0.95
  ) +

  geom_text(
    aes(y = label_y, label = sig_plot, vjust = label_vjust),
    size = 3,
    fontface = "bold",
    colour = "black"
  ) +

  geom_rect(
    data = region_bar_df,
    aes(xmin = xmin, xmax = xmax, ymin = top_ymin, ymax = top_ymax),
    inherit.aes = FALSE,
    fill = "#E6E6E6",
    colour = "white",
    linewidth = 0.35
  ) +

  geom_text(
    data = region_bar_df,
    aes(x = xmid, y = top_label_y, label = area_m),
    inherit.aes = FALSE,
    size = 3.2,
    fontface = "bold",
    colour = "black"
  ) +

  scale_colour_manual(
    values = region_colors,
    name = "Brain region"
  ) +

  scale_fill_manual(
    values = region_colors,
    name = "Brain region"
  ) +

  scale_size_continuous(
    range = c(2.2, 6.0),
    name = expression(-log[10]*"(adjusted P)")
  ) +

  scale_x_continuous(
    breaks = plot_df$x_pos,
    labels = as.character(plot_df$subtype_focus),
    expand = expansion(mult = c(0.02, 0.02))
  ) +

  scale_y_continuous(
    limits = c(-y_lim, top_ymax + 0.15),
    breaks = pretty(c(-y_lim, y_lim), n = 6),
    expand = expansion(mult = c(0.02, 0.02))
  ) +

  labs(
    title = "AD-associated density shifts in DAA and DAM subtypes",
    subtitle = "Top bars denote brain regions; vertical direction denotes AD relative to Control",
    x = NULL,
    y = expression(log[2]*"(AD / Control density)")
  ) +

  theme_nature(base_size = 8) +
  theme(
    legend.position = "right",
    axis.text.x = element_text(size = 8, angle = 45, hjust = 1, vjust = 1),
    axis.text.y = element_text(size = 8),
    axis.title.y = element_text(size = 9),
    plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
    plot.subtitle = element_text(size = 8, hjust = 0.5, colour = "grey35")
  ) +

  guides(
    colour = guide_legend(
      order = 1,
      override.aes = list(size = 4, linewidth = 1.2)
    ),
    fill = "none",
    size = guide_legend(order = 2)
  )

# 5. Save 16 x 8

ggsave(
  filename = file.path(out_dir, "B_lollipop_top_region_bar_no_region_on_x_16x8.pdf"),
  plot = p_top_region_lollipop,
  width = 11,
  height = 5,
  device = cairo_pdf
)


print(p_top_region_lollipop)

cat("Done.\nOutput directory:\n", out_dir, "\n")

## DAA/DAM proportions in digital NVUs

In [ ]:
library(Seurat)
library(dplyr)
library(tidyr)
library(ggplot2)
library(patchwork)
library(rstatix)
library(ggpubr)

# Parameters

AD_samples <- c("D03556C4","D01574C4","C01834C3","D04303A6",
                "D03556E4","D01574B6","D03556D4","D04303D1",
                "D03556E6","C01840B1")

Con_samples <- unique(c("D01574A6","D01574B1",
                         "B03421A5","B03421A6","D03556D6","D03556E2",
                         "D04305A6","D04305A4","C04595E2","C04595F1",
                         "D03556F6","D03556F4","control_1","control_2"))

target_regions     <- c("CA1","SLRM","FAS")
daa_dam_subtypes   <- c("Astro_2","Astro_3","Micro_2")
astro_subtypes_all <- c("Astro_0","Astro_1","Astro_2","Astro_3","Astro_4")
micro_subtypes_all <- c("Micro_0","Micro_1","Micro_2","Micro_3","Micro_4")
disease_colors     <- c("Control" = "#4DBBD5", "AD" = "#E64B35")

NVU = NVU_hip
meta <- NVU@meta.data %>%
  as.data.frame() %>%
  mutate(cellid = rownames(.))

meta_nvu <- meta %>%
  dplyr::filter(
    sample_id %in% c(AD_samples, Con_samples),
    area_m    %in% target_regions,
    !is.na(unit_id), unit_id != ""
  ) %>%
  dplyr::mutate(
    disease_group = case_when(
      sample_id %in% AD_samples  ~ "AD",
      sample_id %in% Con_samples ~ "Control",
      TRUE ~ NA_character_
    ),
    unit_id_full = paste0(sample_id, "_", unit_id)
  ) %>%
  dplyr::filter(!is.na(disease_group))

message("NVU 细胞数: ", nrow(meta_nvu))



unit_has_daa_dam <- meta_nvu %>%
  dplyr::group_by(sample_id, disease_group, area_m, unit_id_full) %>%
  dplyr::summarise(
    has_DAA_DAM     = any(subcelltype %in% daa_dam_subtypes),
    has_DAA         = any(subcelltype %in% c("Astro_2","Astro_3")),
    has_DAM         = any(subcelltype == "Micro_2"),
    has_DAA_AND_DAM = any(subcelltype %in% c("Astro_2","Astro_3")) &  # added
                      any(subcelltype == "Micro_2"),  # added
    .groups         = "drop"
  )

prop_unit_A <- unit_has_daa_dam %>%
  dplyr::group_by(sample_id, disease_group, area_m) %>%
  dplyr::summarise(
    n_total           = n(),
    n_has_daa_dam     = sum(has_DAA_DAM),
    n_has_daa         = sum(has_DAA),
    n_has_dam         = sum(has_DAM),
    n_has_daa_and_dam = sum(has_DAA_AND_DAM),  # added
    .groups           = "drop"
  ) %>%
  dplyr::filter(n_total >= 3) %>%
  dplyr::mutate(
    pct_daa_dam     = n_has_daa_dam     / n_total * 100,
    pct_daa         = n_has_daa         / n_total * 100,
    pct_dam         = n_has_dam         / n_total * 100,
    pct_daa_and_dam = n_has_daa_and_dam / n_total * 100,  # added
    disease_group   = factor(disease_group, levels = c("Control","AD")),
    area_m          = factor(area_m,        levels = target_regions)
  )

plot_dat_A <- prop_unit_A %>%
  tidyr::pivot_longer(
    cols      = c(pct_daa_dam, pct_daa, pct_dam, pct_daa_and_dam),  # added
    names_to  = "metric",
    values_to = "pct"
  ) %>%
  dplyr::mutate(
    metric = factor(metric,
                    levels = c("pct_daa_dam","pct_daa",
                               "pct_dam","pct_daa_and_dam"))  # added
  )


stat_A <- plot_dat_A %>%
  dplyr::group_by(area_m, metric) %>%
  rstatix::wilcox_test(pct ~ disease_group) %>%
  dplyr::ungroup() %>%
  dplyr::group_by(metric) %>%
  rstatix::adjust_pvalue(method = "BH") %>%
  rstatix::add_significance(
    p.col     = "p.adj",
    cutpoints = c(0, 0.001, 0.01, 0.05, 1),
    symbols   = c("***", "**", "*", "ns")
  ) %>%
  dplyr::ungroup()



n_astro_sub <- meta_nvu %>%
  dplyr::filter(subcelltype %in% astro_subtypes_all) %>%
  dplyr::group_by(sample_id, disease_group, area_m) %>%
  dplyr::summarise(n_astro = n(), .groups = "drop")

prop_B <- meta_nvu %>%
  dplyr::filter(subcelltype %in% c("Astro_2","Astro_3")) %>%
  dplyr::group_by(sample_id, disease_group, area_m, subcelltype) %>%
  dplyr::summarise(n_sub = n(), .groups = "drop") %>%
  tidyr::complete(
    tidyr::nesting(sample_id, disease_group, area_m),
    subcelltype = c("Astro_2","Astro_3"),
    fill = list(n_sub = 0)
  ) %>%
  dplyr::left_join(n_astro_sub,
                    by = c("sample_id","disease_group","area_m")) %>%
  dplyr::filter(!is.na(n_astro), n_astro > 0) %>%
  dplyr::mutate(
    prop_pct      = n_sub / n_astro * 100,
    disease_group = factor(disease_group, levels = c("Control","AD")),
    area_m        = factor(area_m,        levels = target_regions),
    subcelltype   = factor(subcelltype,   levels = c("Astro_2","Astro_3"))
  )

stat_B <- prop_B %>%
  dplyr::group_by(area_m, subcelltype) %>%
  rstatix::wilcox_test(prop_pct ~ disease_group) %>%
  rstatix::adjust_pvalue(method = "BH") %>%
  rstatix::add_significance(
    p.col = "p.adj",
    cutpoints = c(0,0.001,0.01,0.05,1),
    symbols   = c("***","**","*","ns")
  ) %>%
  dplyr::ungroup()


n_micro_sub <- meta_nvu %>%
  dplyr::filter(subcelltype %in% micro_subtypes_all) %>%
  dplyr::group_by(sample_id, disease_group, area_m) %>%
  dplyr::summarise(n_micro = n(), .groups = "drop")

prop_C <- meta_nvu %>%
  dplyr::filter(subcelltype == "Micro_2") %>%
  dplyr::group_by(sample_id, disease_group, area_m, subcelltype) %>%
  dplyr::summarise(n_sub = n(), .groups = "drop") %>%
  tidyr::complete(
    tidyr::nesting(sample_id, disease_group, area_m),
    subcelltype = "Micro_2",
    fill = list(n_sub = 0)
  ) %>%
  dplyr::left_join(n_micro_sub,
                    by = c("sample_id","disease_group","area_m")) %>%
  dplyr::filter(!is.na(n_micro), n_micro > 0) %>%
  dplyr::mutate(
    prop_pct      = n_sub / n_micro * 100,
    disease_group = factor(disease_group, levels = c("Control","AD")),
    area_m        = factor(area_m,        levels = target_regions),
    subcelltype   = factor(subcelltype,   levels = "Micro_2")
  )

stat_C <- prop_C %>%
  dplyr::group_by(area_m, subcelltype) %>%
  rstatix::wilcox_test(prop_pct ~ disease_group) %>%
  rstatix::adjust_pvalue(method = "BH") %>%
  rstatix::add_significance(
    p.col = "p.adj",
    cutpoints = c(0,0.001,0.01,0.05,1),
    symbols   = c("***","**","*","ns")
  ) %>%
  dplyr::ungroup()

In [ ]:
make_plot <- function(dat, y_col, facet_row_col, facet_row_labels,
                      stat_df, stat_group_cols,
                      title_str, y_lab) {

  dat2 <- dat %>%
    dplyr::rename(y_val = all_of(y_col)) %>%
    dplyr::mutate(
      disease_group = factor(disease_group, levels = c("Control","AD"))
    )

  y_max_df <- dat2 %>%
    dplyr::group_by(across(all_of(c("area_m", facet_row_col)))) %>%
    dplyr::summarise(y_max = max(y_val, na.rm = TRUE) * 1.18,
                     .groups = "drop")

  sig_df <- stat_df %>%
    dplyr::filter(p.adj.signif != "ns") %>%
    dplyr::left_join(y_max_df, by = c("area_m", facet_row_col)) %>%
    dplyr::mutate(
      x1    = 1L,
      x2    = 2L,
      x_mid = 1.5
    )

  row_sym <- rlang::sym(facet_row_col)

  p <- ggplot(dat2, aes(x = disease_group, y = y_val)) +
    geom_boxplot(
      aes(fill = disease_group),
      width = 0.55, alpha = 0.3, outlier.shape = NA,
      linewidth = 0.45, color = "black"
    ) +
    geom_jitter(
      aes(fill = disease_group),
      width = 0.15, height = 0,
      size = 1.8, alpha = 0.85,
      shape = 21, stroke = 0.3, color = "white"
    ) +
    facet_grid(
      rows     = vars(!!row_sym),
      cols     = vars(area_m),
      labeller = labeller(.rows = facet_row_labels),
      scales   = "free_y"
    ) +
    scale_fill_manual(values = disease_colors) +
    scale_y_continuous(expand = expansion(mult = c(0.03, 0.2))) +
    labs(title = title_str, x = NULL, y = y_lab) +
    theme_classic(base_size = 11) +
    theme(
      plot.title       = element_text(face = "bold", size = 11, hjust = 0),
      strip.background = element_blank(),
      strip.text.x     = element_text(face = "bold", size = 10),
      strip.text.y     = element_text(face = "bold", size = 9,
                                       angle = 0, hjust = 0),
      axis.text        = element_text(size = 9, color = "black"),
      axis.line        = element_line(linewidth = 0.4),
      axis.ticks       = element_line(linewidth = 0.35),
      legend.position  = "none",
      panel.spacing    = unit(0.4, "cm"),
      plot.margin      = margin(6, 6, 6, 6)
    )

  if (nrow(sig_df) > 0) {
    p <- p +
      geom_segment(
        data = sig_df,
        aes(x = x1, xend = x2,
            y = y_max, yend = y_max),
        inherit.aes = FALSE,
        linewidth = 0.4, color = "black"
      ) +
      geom_segment(
        data = sig_df,
        aes(x = x1, xend = x1,
            y = y_max, yend = y_max * 0.975),
        inherit.aes = FALSE,
        linewidth = 0.4, color = "black"
      ) +
      geom_segment(
        data = sig_df,
        aes(x = x2, xend = x2,
            y = y_max, yend = y_max * 0.975),
        inherit.aes = FALSE,
        linewidth = 0.4, color = "black"
      ) +
      geom_text(
        data = sig_df,
        aes(x = x_mid, y = y_max * 1.03,
            label = p.adj.signif),
        inherit.aes = FALSE,
        size = 3.5, color = "black"
      )
  }
  p
}

metric_labels <- c(
  pct_daa_dam     = "DAA or DAM",
  pct_daa         = "DAA (Astro_2/3)",
  pct_dam         = "DAM (Micro_2)",
  pct_daa_and_dam = "DAA and DAM"  # added
)

p_A <- make_plot(
  dat              = plot_dat_A,
  y_col            = "pct",
  facet_row_col    = "metric",
  facet_row_labels = metric_labels,
  stat_df          = stat_A,
  stat_group_cols  = c("area_m","metric"),
  title_str        = "A  |  NVU units containing DAA/DAM cells",
  y_lab            = "% of NVU units"
)

p_B <- make_plot(
  dat              = prop_B,
  y_col            = "prop_pct",
  facet_row_col    = "subcelltype",
  facet_row_labels = c("Astro_2" = "Astro_2 (DAA)",
                        "Astro_3" = "Astro_3 (DAA)"),
  stat_df          = stat_B,
  stat_group_cols  = c("area_m","subcelltype"),
  title_str        = "B  |  DAA proportion of subtyped Astro",
  y_lab            = "% of subtyped Astro"
)

p_C <- make_plot(
  dat              = prop_C,
  y_col            = "prop_pct",
  facet_row_col    = "subcelltype",
  facet_row_labels = c("Micro_2" = "Micro_2 (DAM)"),
  stat_df          = stat_C,
  stat_group_cols  = c("area_m","subcelltype"),
  title_str        = "C  |  DAM proportion of subtyped Micro",
  y_lab            = "% of subtyped Micro"
)

fig <- (p_A) +
  plot_annotation(
    title   = expression("DAA/DAM in NVU — AD vs Control"),
    caption = paste0(
      "A: % of NVU units containing ≥1 DAA/DAM cell  |  ",
      "B: % of subtyped Astro  |  C: % of subtyped Micro\n",
      "Wilcoxon; BH-adjusted *p<0.05 **p<0.01 ***p<0.001"
    ),
    theme = theme(
      plot.title   = element_text(face = "bold", size = 13, hjust = 0.5),
      plot.caption = element_text(size = 8, color = "grey50", hjust = 0)
    )
  )

cairo_pdf("DAA_DAM_proportion_final.pdf", width = 10, height = 10)
print(fig)
dev.off()

ggsave("DAA_DAM_proportion_final.png", fig,
       width = 10, height = 8, dpi = 300)

message("完成")

In [ ]:

library(Seurat)
library(dplyr)
library(tidyr)
library(ggplot2)
library(patchwork)
library(rstatix)

# Parameters

target_regions_cortex <- c("L23","L456")
daa_dam_subtypes      <- c("Astro_2","Astro_3","Micro_2")
disease_colors        <- c("Control" = "#4DBBD5", "AD" = "#E64B35")
exclude_sample        <- "GSM8330064_B02008C6"


meta_cortex <- NVU_cortex@meta.data %>%
  as.data.frame() %>%
  mutate(cellid = rownames(.))

meta_nvu_cortex <- meta_cortex %>%
  dplyr::filter(
    area_m    %in% target_regions_cortex,
    !is.na(unit_id), unit_id != "",
    sample_id != exclude_sample
  ) %>%
  dplyr::mutate(
    disease_group = case_when(
      group == "AD"      ~ "AD",
      group == "Control" ~ "Control",
      group == "Con"     ~ "Control",
      TRUE               ~ as.character(group)
    ),
    unit_id_full = paste0(sample_id, "_", unit_id)
  ) %>%
  dplyr::filter(disease_group %in% c("AD","Control"))

message("过滤后皮层 NVU 细胞数: ", nrow(meta_nvu_cortex))
message("\n各样本细胞数:")


unit_cortex <- meta_nvu_cortex %>%
  dplyr::group_by(sample_id, disease_group, area_m, unit_id_full) %>%
  dplyr::summarise(
    has_DAA_DAM     = any(subcelltype %in% daa_dam_subtypes),
    has_DAA         = any(subcelltype %in% c("Astro_2","Astro_3")),
    has_DAM         = any(subcelltype == "Micro_2"),
    has_DAA_AND_DAM = any(subcelltype %in% c("Astro_2","Astro_3")) &  # added
                      any(subcelltype == "Micro_2"),  # added
    .groups         = "drop"
  )

prop_cortex <- unit_cortex %>%
  dplyr::group_by(sample_id, disease_group, area_m) %>%
  dplyr::summarise(
    n_total           = n(),
    n_has_daa_dam     = sum(has_DAA_DAM),
    n_has_daa         = sum(has_DAA),
    n_has_dam         = sum(has_DAM),
    n_has_daa_and_dam = sum(has_DAA_AND_DAM),  # added
    .groups           = "drop"
  ) %>%
  dplyr::filter(n_total >= 3) %>%
  dplyr::mutate(
    pct_daa_dam     = n_has_daa_dam     / n_total * 100,
    pct_daa         = n_has_daa         / n_total * 100,
    pct_dam         = n_has_dam         / n_total * 100,
    pct_daa_and_dam = n_has_daa_and_dam / n_total * 100,  # added
    disease_group   = factor(disease_group, levels = c("Control","AD")),
    area_m          = factor(area_m, levels = target_regions_cortex)
  )

message("\n各组 sample 数:")

plot_dat_cortex <- prop_cortex %>%
  tidyr::pivot_longer(
    cols      = c(pct_daa_dam, pct_daa, pct_dam, pct_daa_and_dam),  # added
    names_to  = "metric",
    values_to = "pct"
  ) %>%
  dplyr::mutate(
    metric = factor(metric,
                    levels = c("pct_daa_dam","pct_daa",
                               "pct_dam","pct_daa_and_dam"))  # added
  )


stat_cortex <- plot_dat_cortex %>%
  dplyr::group_by(area_m, metric) %>%
  rstatix::wilcox_test(pct ~ disease_group) %>%
  dplyr::ungroup() %>%
  dplyr::group_by(metric) %>%
  rstatix::adjust_pvalue(method = "BH") %>%
  rstatix::add_significance(
    p.col     = "p.adj",
    cutpoints = c(0, 0.001, 0.01, 0.05, 1),
    symbols   = c("***","**","*","ns")
  ) %>%
  dplyr::ungroup()

message("\n统计结果:")
print(stat_cortex)


y_max_df <- plot_dat_cortex %>%
  dplyr::group_by(area_m, metric) %>%
  dplyr::summarise(y_max = max(pct, na.rm = TRUE) * 1.18,
                   .groups = "drop") %>%
  dplyr::mutate(across(c(area_m, metric), as.character))

sig_df <- stat_cortex %>%
  dplyr::filter(p.adj.signif != "ns") %>%
  dplyr::mutate(across(c(area_m, metric), as.character)) %>%
  dplyr::left_join(y_max_df, by = c("area_m","metric")) %>%
  dplyr::mutate(x1 = 1L, x2 = 2L, x_mid = 1.5)

message("显著标注行数: ", nrow(sig_df))

# Plotting

metric_labels <- c(
  pct_daa_dam     = "DAA or DAM",
  pct_daa         = "DAA (Astro_2/3)",
  pct_dam         = "DAM (Micro_2)",
  pct_daa_and_dam = "DAA and DAM"  # added
)

p_cortex <- ggplot(plot_dat_cortex,
                    aes(x = disease_group, y = pct)) +
  geom_boxplot(
    aes(fill = disease_group),
    width         = 0.55,
    alpha         = 0.3,
    outlier.shape = NA,
    linewidth     = 0.45,
    color         = "black"
  ) +
  geom_jitter(
    aes(fill = disease_group),
    width  = 0.15,
    height = 0,
    size   = 1.8,
    alpha  = 0.85,
    shape  = 21,
    stroke = 0.3,
    color  = "white"
  ) +
  {if (nrow(sig_df) > 0) list(
    geom_segment(
      data        = sig_df,
      aes(x = x1, xend = x2, y = y_max, yend = y_max),
      inherit.aes = FALSE,
      linewidth   = 0.4,
      color       = "black"
    ),
    geom_segment(
      data        = sig_df,
      aes(x = x1, xend = x1, y = y_max, yend = y_max * 0.975),
      inherit.aes = FALSE,
      linewidth   = 0.4,
      color       = "black"
    ),
    geom_segment(
      data        = sig_df,
      aes(x = x2, xend = x2, y = y_max, yend = y_max * 0.975),
      inherit.aes = FALSE,
      linewidth   = 0.4,
      color       = "black"
    ),
    geom_text(
      data        = sig_df,
      aes(x = x_mid, y = y_max * 1.03, label = p.adj.signif),
      inherit.aes = FALSE,
      size        = 3.5,
      color       = "black"
    )
  )} +
  facet_grid(
    rows     = vars(metric),
    cols     = vars(area_m),
    labeller = labeller(metric = metric_labels),
    scales   = "free_y"
  ) +
  scale_fill_manual(values = disease_colors) +
  scale_y_continuous(expand = expansion(mult = c(0.03, 0.2))) +
  labs(
    title   = expression("NVU units containing DAA/DAM — cortex (AD vs Control)"),
    caption = paste0(
      "Excluded: ", exclude_sample, "\n",
      "Wilcoxon; BH-adjusted per metric  *p<0.05 **p<0.01 ***p<0.001"
    ),
    x = NULL,
    y = "% of NVU units"
  ) +
  theme_classic(base_size = 11) +
  theme(
    plot.title       = element_text(face = "bold", size = 12, hjust = 0.5),
    plot.caption     = element_text(size = 8, color = "grey50", hjust = 0),
    strip.background = element_blank(),
    strip.text.x     = element_text(face = "bold", size = 11),
    strip.text.y     = element_text(face = "bold", size = 9,
                                     angle = 0, hjust = 0),
    axis.text        = element_text(size = 9, color = "black"),
    axis.line        = element_line(linewidth = 0.4),
    axis.ticks       = element_line(linewidth = 0.35),
    legend.position  = "none",
    panel.spacing    = unit(0.5, "cm"),
    plot.margin      = margin(8, 8, 8, 8)
  )

# Save

cairo_pdf("DAA_DAM_cortex_L23_L456.pdf", width = 8, height = 10)
print(p_cortex)
dev.off()

ggsave("DAA_DAM_cortex_L23_L456.png", p_cortex,
       width = 8, height = 8, dpi = 300)

write.csv(prop_cortex,  "DAA_DAM_cortex_prop.csv",  row.names = FALSE)
write.csv(stat_cortex,  "DAA_DAM_cortex_stats.csv", row.names = FALSE)

message("完成: DAA_DAM_cortex_L23_L456.pdf/.png")